# 01 Prepare Roboflow Field2 Full Pack

00 노트북에서 확인한 결과, YOLO 표지판 탐지 v1은 field2 데이터만으로 시작하는 것이 가장 깔끔함.

이번 01 노트북의 목적은 field2 표지판 6클래스 전체 2,542장을 Roboflow 업로드용으로 정리하는 것임.

생성물은 두 종류임.

1. 전체 zip
   - `roboflow_sign_field2_full_v1_all_flat.zip`
   - 한 번에 전체 이미지를 Roboflow 프로젝트에 올릴 때 사용함.

2. 클래스별 zip
   - `field2_left.zip`
   - `field2_right.zip`
   - `field2_straight.zip`
   - `field2_horn.zip`
   - `field2_stop.zip`
   - `field2_speed_20.zip`
   - 팀원별로 클래스 batch를 나눠 업로드/할당하고 싶을 때 사용함.

주의: zip 안에 클래스명이 들어가 있어도 Roboflow가 자동으로 bbox/class 라벨을 만들어주지는 않음.  
파일명과 zip 이름은 라벨링할 사람이 참고하는 힌트이고, 실제 YOLO label은 Roboflow에서 bounding box를 직접 그려 생성함.

## 0. 실행 설정

이 노트북은 local VSCode `<env>` kernel에서 실행함.

기본 정책:

- field2 데이터만 사용함.
- 6개 표지판 클래스만 사용함.
- 00 노트북에서 생성한 manifest 중 `read_ok=True`만 사용함.
- background/traffic 데이터는 이번 pack에 넣지 않음.

In [1]:
from pathlib import Path
import os
import json
import shutil
import zipfile
from datetime import datetime

import cv2
import numpy as np
import pandas as pd

PROJECT_ROOT = Path(r'~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization')
EXP_ROOT = PROJECT_ROOT / '10_experiments' / '07_sign_detection_yolo'

AUDIT_TABLE_DIR = EXP_ROOT / 'review_outputs' / '00_data_audit' / 'tables'
SIGN_MANIFEST_CSV = AUDIT_TABLE_DIR / 'sign_labeling_manifest.csv'

PACK_ROOT = EXP_ROOT / 'labeling_packs' / 'roboflow_sign_field2_full_v1'
FLAT_IMAGES_DIR = PACK_ROOT / 'images_flat'
BY_CLASS_DIR = PACK_ROOT / 'images_by_class'
ZIP_DIR = PACK_ROOT / 'zips'
TABLE_DIR = PACK_ROOT / 'tables'
DOC_DIR = PACK_ROOT / 'docs'

SIGN_CLASSES = ['left', 'right', 'straight', 'horn', 'stop', 'speed_20']

# True이면 기존 pack 폴더를 삭제하고 다시 만듦.
# 같은 pack을 다시 만들 때만 True로 둔다.
REBUILD_PACK = True

print('SIGN_MANIFEST_CSV:', SIGN_MANIFEST_CSV, SIGN_MANIFEST_CSV.exists())
print('PACK_ROOT:', PACK_ROOT)
assert SIGN_MANIFEST_CSV.exists(), '먼저 00 노트북을 실행해서 sign_labeling_manifest.csv를 만들어야 함.'

SIGN_MANIFEST_CSV: ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\07_sign_detection_yolo\review_outputs\00_data_audit\tables\sign_labeling_manifest.csv True
PACK_ROOT: ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\07_sign_detection_yolo\labeling_packs\roboflow_sign_field2_full_v1


## 1. 00 manifest 읽고 field2-only 필터링

필터 기준:

```text
candidate_type == sign_labeling
source == field2
target_class in 6 classes
read_ok == True
```

예상 개수는 2,542장임.

In [2]:
manifest_df = pd.read_csv(SIGN_MANIFEST_CSV)

field2_df = manifest_df[
    (manifest_df['candidate_type'] == 'sign_labeling') &
    (manifest_df['source'] == 'field2') &
    (manifest_df['target_class'].isin(SIGN_CLASSES)) &
    (manifest_df['read_ok'].astype(str).str.lower() == 'true')
].copy()

field2_df = field2_df.sort_values(['target_class', 'file_name']).reset_index(drop=True)

print('field2 sign rows:', len(field2_df))
display(field2_df.head(10))
display(field2_df.groupby('target_class').size().reindex(SIGN_CLASSES).reset_index(name='count'))

assert len(field2_df) == 2542, f'예상 field2 sign count 2542와 다름: {len(field2_df)}'
assert field2_df['target_class'].isna().sum() == 0

field2 sign rows: 2542


,source,candidate_type,folder_label,target_class,absolute_path,file_name,rel_to_source_folder,source_root,note,read_ok,width,height,channels
0,field2,sign_labeling,horn,horn,~\02_Projects\University\26-1_Em...,20260501_122048_06_sign_capture_drive_horn_099...,20260501_122048_06_sign_capture_drive_horn_099...,~\02_Projects\University\26-1_Em...,field2 moving sign capture,True,1296.0,972.0,3.0
1,field2,sign_labeling,horn,horn,~\02_Projects\University\26-1_Em...,20260501_122048_06_sign_capture_drive_horn_100...,20260501_122048_06_sign_capture_drive_horn_100...,~\02_Projects\University\26-1_Em...,field2 moving sign capture,True,1296.0,972.0,3.0
2,field2,sign_labeling,horn,horn,~\02_Projects\University\26-1_Em...,20260501_122049_06_sign_capture_drive_horn_100...,20260501_122049_06_sign_capture_drive_horn_100...,~\02_Projects\University\26-1_Em...,field2 moving sign capture,True,1296.0,972.0,3.0
3,field2,sign_labeling,horn,horn,~\02_Projects\University\26-1_Em...,20260501_122049_06_sign_capture_drive_horn_100...,20260501_122049_06_sign_capture_drive_horn_100...,~\02_Projects\University\26-1_Em...,field2 moving sign capture,True,1296.0,972.0,3.0
4,field2,sign_labeling,horn,horn,~\02_Projects\University\26-1_Em...,20260501_122049_06_sign_capture_drive_horn_100...,20260501_122049_06_sign_capture_drive_horn_100...,~\02_Projects\University\26-1_Em...,field2 moving sign capture,True,1296.0,972.0,3.0
5,field2,sign_labeling,horn,horn,~\02_Projects\University\26-1_Em...,20260501_122049_06_sign_capture_drive_horn_100...,20260501_122049_06_sign_capture_drive_horn_100...,~\02_Projects\University\26-1_Em...,field2 moving sign capture,True,1296.0,972.0,3.0
6,field2,sign_labeling,horn,horn,~\02_Projects\University\26-1_Em...,20260501_122050_06_sign_capture_drive_horn_100...,20260501_122050_06_sign_capture_drive_horn_100...,~\02_Projects\University\26-1_Em...,field2 moving sign capture,True,1296.0,972.0,3.0
7,field2,sign_labeling,horn,horn,~\02_Projects\University\26-1_Em...,20260501_122050_06_sign_capture_drive_horn_100...,20260501_122050_06_sign_capture_drive_horn_100...,~\02_Projects\University\26-1_Em...,field2 moving sign capture,True,1296.0,972.0,3.0
8,field2,sign_labeling,horn,horn,~\02_Projects\University\26-1_Em...,20260501_122050_06_sign_capture_drive_horn_100...,20260501_122050_06_sign_capture_drive_horn_100...,~\02_Projects\University\26-1_Em...,field2 moving sign capture,True,1296.0,972.0,3.0
9,field2,sign_labeling,horn,horn,~\02_Projects\University\26-1_Em...,20260501_122051_06_sign_capture_drive_horn_100...,20260501_122051_06_sign_capture_drive_horn_100...,~\02_Projects\University\26-1_Em...,field2 moving sign capture,True,1296.0,972.0,3.0


,target_class,count
0,left,477
1,right,412
2,straight,386
3,horn,457
4,stop,395
5,speed_20,415


## 2. Pack 폴더 초기화

`REBUILD_PACK=True`이면 기존 pack 폴더를 지우고 다시 생성함.  
Roboflow에 이미 업로드한 zip이 있다면 덮어써도 되는지 확인하고 실행해야 함.

In [3]:
if PACK_ROOT.exists() and REBUILD_PACK:
    shutil.rmtree(PACK_ROOT)

for p in [FLAT_IMAGES_DIR, BY_CLASS_DIR, ZIP_DIR, TABLE_DIR, DOC_DIR]:
    p.mkdir(parents=True, exist_ok=True)

for cls in SIGN_CLASSES:
    (BY_CLASS_DIR / cls).mkdir(parents=True, exist_ok=True)

print('pack dirs ready')
print('FLAT_IMAGES_DIR:', FLAT_IMAGES_DIR)
print('BY_CLASS_DIR:', BY_CLASS_DIR)
print('ZIP_DIR:', ZIP_DIR)

pack dirs ready
FLAT_IMAGES_DIR: ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\07_sign_detection_yolo\labeling_packs\roboflow_sign_field2_full_v1\images_flat
BY_CLASS_DIR: ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\07_sign_detection_yolo\labeling_packs\roboflow_sign_field2_full_v1\images_by_class
ZIP_DIR: ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\07_sign_detection_yolo\labeling_packs\roboflow_sign_field2_full_v1\zips


## 3. 이미지 복사 및 파일명 정리

Roboflow에서 팀원이 파일명만 봐도 힌트를 얻을 수 있도록 이름을 통일함.

예:

```text
field2_left_000001.jpg
field2_stop_000001.jpg
field2_speed_20_000001.jpg
```

동일 이미지는 두 위치에 복사함.

- `images_flat/`: 전체 zip용
- `images_by_class/{class}/`: 클래스별 zip용

In [4]:
def win_long_path(path):
    p = Path(path)
    s = str(p if p.is_absolute() else p.resolve())
    if os.name == 'nt' and not s.startswith('\\\\?\\'):
        return '\\\\?\\' + s
    return s


def copy_file(src: Path, dst: Path):
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(win_long_path(src), win_long_path(dst))


pack_rows = []

for cls in SIGN_CLASSES:
    cls_df = field2_df[field2_df['target_class'] == cls].sort_values('file_name').reset_index(drop=True)
    for idx, row in cls_df.iterrows():
        src = Path(row['absolute_path'])
        new_name = f'field2_{cls}_{idx+1:06d}.jpg'
        flat_dst = FLAT_IMAGES_DIR / new_name
        class_dst = BY_CLASS_DIR / cls / new_name

        copy_file(src, flat_dst)
        copy_file(src, class_dst)

        pack_rows.append({
            'source': 'field2',
            'target_class_hint': cls,
            'new_file_name': new_name,
            'flat_image_path': str(flat_dst),
            'class_image_path': str(class_dst),
            'original_path': str(src),
            'original_file_name': row['file_name'],
            'original_folder_label': row['folder_label'],
            'width': row.get('width', ''),
            'height': row.get('height', ''),
            'note': 'Class is only a filename hint. Draw bbox manually in Roboflow.',
        })

pack_df = pd.DataFrame(pack_rows)
UPLOAD_MANIFEST_CSV = TABLE_DIR / 'roboflow_sign_field2_full_v1_upload_manifest.csv'
pack_df.to_csv(UPLOAD_MANIFEST_CSV, index=False, encoding='utf-8-sig')

class_counts = pack_df.groupby('target_class_hint').size().reindex(SIGN_CLASSES).reset_index(name='count')
CLASS_COUNTS_CSV = TABLE_DIR / 'roboflow_sign_field2_full_v1_class_counts.csv'
class_counts.to_csv(CLASS_COUNTS_CSV, index=False, encoding='utf-8-sig')

print('copied images:', len(pack_df))
display(class_counts)
print('saved:', UPLOAD_MANIFEST_CSV)
print('saved:', CLASS_COUNTS_CSV)

copied images: 2542


,target_class_hint,count
0,left,477
1,right,412
2,straight,386
3,horn,457
4,stop,395
5,speed_20,415


saved: ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\07_sign_detection_yolo\labeling_packs\roboflow_sign_field2_full_v1\tables\roboflow_sign_field2_full_v1_upload_manifest.csv
saved: ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\07_sign_detection_yolo\labeling_packs\roboflow_sign_field2_full_v1\tables\roboflow_sign_field2_full_v1_class_counts.csv


## 4. 복사 결과 검증

복사된 이미지 개수와 파일 크기를 확인함.  
0바이트 파일이 없어야 함.

In [5]:
flat_files = sorted(FLAT_IMAGES_DIR.glob('*.jpg'))
class_files = sorted(BY_CLASS_DIR.glob('*/*.jpg'))

print('flat files:', len(flat_files))
print('class files:', len(class_files))
assert len(flat_files) == len(pack_df)
assert len(class_files) == len(pack_df)

zero_flat = [p for p in flat_files if p.stat().st_size == 0]
zero_class = [p for p in class_files if p.stat().st_size == 0]
print('zero_flat:', len(zero_flat))
print('zero_class:', len(zero_class))
assert not zero_flat
assert not zero_class

size_summary = pd.DataFrame({
    'bytes': [p.stat().st_size for p in flat_files],
})
display(size_summary.describe())

flat files: 2542
class files: 2542
zero_flat: 0
zero_class: 0


,bytes
count,2542.000000
mean,165302.200629
std,15358.681825
min,86589.000000
25%,151536.500000
50%,164591.500000
75%,177639.750000
max,213013.000000


## 5. 라벨링 가이드 문서 생성

Roboflow의 Project Instructions나 팀원 공유용으로 그대로 붙여 넣을 수 있는 짧은 가이드를 생성함.

In [6]:
instructions = f'''# Roboflow Sign Labeling Instructions

Project type: Object Detection

Classes:
- left
- right
- straight
- horn
- stop
- speed_20

Labeling rules:
1. Draw one bounding box around the visible sign plate only.
2. Do not include the pole or base in the box.
3. Use the filename as a class hint, but verify the actual sign visually.
4. If the sign is too small or too blurry to identify, skip the image or mark it for review.
5. If multiple project signs are clearly visible, label all visible signs.
6. Red/green traffic lights are not part of this 6-class sign project.
7. Background images are not included in this v1 pack.

Current pack:
- source: field2 only
- images: {len(pack_df)}
- created_at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
'''

INSTRUCTIONS_MD = DOC_DIR / 'roboflow_labeling_instructions.md'
INSTRUCTIONS_MD.write_text(instructions, encoding='utf-8')
print(instructions)
print('saved:', INSTRUCTIONS_MD)

# Roboflow Sign Labeling Instructions

Project type: Object Detection

Classes:
- left
- right
- straight
- horn
- stop
- speed_20

Labeling rules:
1. Draw one bounding box around the visible sign plate only.
2. Do not include the pole or base in the box.
3. Use the filename as a class hint, but verify the actual sign visually.
4. If the sign is too small or too blurry to identify, skip the image or mark it for review.
5. If multiple project signs are clearly visible, label all visible signs.
6. Red/green traffic lights are not part of this 6-class sign project.
7. Background images are not included in this v1 pack.

Current pack:
- source: field2 only
- images: 2542
- created_at: 2026-05-06 17:03:14

saved: ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\07_sign_detection_yolo\labeling_packs\roboflow_sign_field2_full_v1\docs\roboflow_labeling_instructions.md


## 6. Zip 생성

두 종류의 zip을 생성함.

- 전체 flat zip: 한 번에 프로젝트에 업로드할 때 사용
- 클래스별 zip: 클래스별 batch를 나눠 업로드하거나 팀원별 작업 배정을 쉽게 하고 싶을 때 사용

Roboflow에 클래스별 zip을 모두 같은 프로젝트에 순서대로 업로드해도 됨.

In [7]:
def make_zip_from_files(zip_path: Path, files, arcname_fn):
    if zip_path.exists():
        zip_path.unlink()
    with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
        for p in files:
            zf.write(win_long_path(p), arcname=arcname_fn(p))
    return zip_path


ALL_FLAT_ZIP = ZIP_DIR / 'roboflow_sign_field2_full_v1_all_flat.zip'
make_zip_from_files(
    ALL_FLAT_ZIP,
    flat_files,
    arcname_fn=lambda p: p.name,
)

class_zip_rows = []
for cls in SIGN_CLASSES:
    files = sorted((BY_CLASS_DIR / cls).glob('*.jpg'))
    zip_path = ZIP_DIR / f'field2_{cls}.zip'
    make_zip_from_files(
        zip_path,
        files,
        arcname_fn=lambda p: p.name,
    )
    class_zip_rows.append({
        'target_class_hint': cls,
        'image_count': len(files),
        'zip_path': str(zip_path),
        'zip_size_mb': round(zip_path.stat().st_size / (1024 * 1024), 2),
    })

zip_rows = [{
    'zip_type': 'all_flat',
    'target_class_hint': 'all',
    'image_count': len(flat_files),
    'zip_path': str(ALL_FLAT_ZIP),
    'zip_size_mb': round(ALL_FLAT_ZIP.stat().st_size / (1024 * 1024), 2),
}] + [
    {'zip_type': 'class_wise', **row}
    for row in class_zip_rows
]

zip_df = pd.DataFrame(zip_rows)
ZIP_INDEX_CSV = TABLE_DIR / 'roboflow_sign_field2_full_v1_zip_index.csv'
zip_df.to_csv(ZIP_INDEX_CSV, index=False, encoding='utf-8-sig')

display(zip_df)
print('saved:', ZIP_INDEX_CSV)

,zip_type,target_class_hint,image_count,zip_path,zip_size_mb
0,all_flat,all,2542,~\02_Projects\University\26-1_Em...,399.78
1,class_wise,left,477,~\02_Projects\University\26-1_Em...,75.35
2,class_wise,right,412,~\02_Projects\University\26-1_Em...,67.09
3,class_wise,straight,386,~\02_Projects\University\26-1_Em...,60.36
4,class_wise,horn,457,~\02_Projects\University\26-1_Em...,70.68
5,class_wise,stop,395,~\02_Projects\University\26-1_Em...,61.87
6,class_wise,speed_20,415,~\02_Projects\University\26-1_Em...,64.44


saved: ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\07_sign_detection_yolo\labeling_packs\roboflow_sign_field2_full_v1\tables\roboflow_sign_field2_full_v1_zip_index.csv


## 7. Roboflow 업로드 방식 선택

### 방식 A: 전체 zip 하나 업로드

`roboflow_sign_field2_full_v1_all_flat.zip`을 업로드함.

장점:

- 한 번에 끝남.
- 모든 이미지가 같은 프로젝트에 들어감.

단점:

- Roboflow UI에서 클래스별 작업 배정이 불편할 수 있음.

### 방식 B: 클래스별 zip 6개 업로드

아래 6개 zip을 같은 프로젝트에 순서대로 업로드함.

```text
field2_left.zip
field2_right.zip
field2_straight.zip
field2_horn.zip
field2_stop.zip
field2_speed_20.zip
```

장점:

- 업로드 batch가 클래스별로 나뉘므로 팀원 배정이 쉬움.
- 각 팀원이 맡은 batch만 라벨링하기 편함.

단점:

- 업로드를 6번 해야 함.

현재 목적이 팀원 분담 라벨링이므로, 우선은 **방식 B: 클래스별 zip 업로드**를 추천함.

## 8. 실행 결과 메모

아직 미실행 상태. 실행 후 아래를 확인하면 됨.

1. `tables/roboflow_sign_field2_full_v1_class_counts.csv`
2. `tables/roboflow_sign_field2_full_v1_upload_manifest.csv`
3. `zips/roboflow_sign_field2_full_v1_all_flat.zip`
4. `zips/field2_left.zip` 등 클래스별 zip 6개
5. `docs/roboflow_labeling_instructions.md`

Roboflow에서는 Object Detection project에 업로드하고, 팀원에게 Labeler 권한을 준 뒤 Assign Images에서 batch를 나눠 배정함.